# Can we predict a warranty claim within a year?
Target: `claim_within_year` (base rate ~12.6%, imbalanced). Following `checklist.md`.

Data notes: no date column, so no time ordering is available and a temporal split is impossible. `device_serial` is unique per row (an identifier, dropped; no entity repeats, verified below), so a stratified random split is the appropriate structure. All remaining columns (channel, age, price, daily hours) are recorded at sale/registration, i.e. known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_fscore_support
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, df.device_serial.is_unique, df.isna().sum().sum())
df.describe(include='all').T

## 1. Set aside the test set (before any modelling)
Serials are unique, so rows are independent entities; stratified 80/20 split.

In [ ]:
y=df['claim_within_year']
X=df.drop(columns=['claim_within_year','device_serial'])   # identifier dropped
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(Xtr),len(Xte),ytr.mean(),yte.mean())

## 2-4. Candidate models (named before scoring)
1. `baseline`: predicts the training prior (trivial)
2. `logreg`: logistic regression, C in {0.01, 0.1, 1}
3. `rf`: random forest, min_samples_leaf in {5, 20, 50}
4. `hgb`: histogram gradient boosting, learning_rate 0.05, max_depth in {2, 3}, max_iter 100

Selection metric: mean ROC AUC across 5-fold stratified CV on training rows only. Preprocessing (scaling, one-hot) lives inside the pipeline, so it is fit per fold. Decision threshold: maximises F1 on out-of-fold CV predictions of the chosen model.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def pre(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
def mk(name):
    kind,p=name.split(':',1)
    if kind=='baseline': return Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))])
    if kind=='logreg': return Pipeline([('p',pre(True)),('m',LogisticRegression(C=float(p),max_iter=1000))])
    if kind=='rf': return Pipeline([('p',pre(False)),('m',RandomForestClassifier(300,min_samples_leaf=int(p),random_state=SEED,n_jobs=-1))])
    if kind=='hgb': return Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(learning_rate=0.05,max_depth=int(p),max_iter=100,random_state=SEED))])
names=['baseline:0']+['logreg:'+s for s in ('0.01','0.1','1')]+['rf:'+s for s in ('5','20','50')]+['hgb:2','hgb:3']
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
oof={};rows=[]
for n in names:
    p=cross_val_predict(mk(n),Xtr,ytr,cv=cv,method='predict_proba')[:,1]
    oof[n]=p
    fa=[roc_auc_score(ytr.iloc[v],p[v]) for _,v in cv.split(Xtr,ytr)]
    rows.append((n,np.mean(fa),np.std(fa),average_precision_score(ytr,p)))
res=pd.DataFrame(rows,columns=['model','cv_auc','auc_sd','cv_pr_auc']).sort_values('cv_auc',ascending=False)
res

## 5. Choice, made on CV only (recorded before touching the test set)

In [ ]:
best=res.iloc[0].model
p=oof[best]
ths=np.linspace(0.02,0.6,59)
f1s=[f1_score(ytr,p>=t,zero_division=0) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print(f'CHOSEN MODEL: {best}; threshold {thr:.2f}')

## 6. Refit on all training rows; score the test set once (headline)

In [ ]:
final=mk(best).fit(Xtr,ytr)
pt=final.predict_proba(Xte)[:,1]
pr,rc,f,_=precision_recall_fscore_support(yte,pt>=thr,average='binary',zero_division=0)
print(f'TEST ROC AUC {roc_auc_score(yte,pt):.3f} | PR AUC {average_precision_score(yte,pt):.3f} (base rate {yte.mean():.3f})')
print(f'at threshold {thr:.2f}: precision {pr:.3f} recall {rc:.3f} F1 {f:.3f}')

## Conclusion
See the printed headline above. Interpret ROC AUC against 0.5 (the baseline) and PR AUC against the claim base rate. The test set was scored exactly once; no choices were revisited afterwards. Limitation: no timestamps, so we cannot confirm performance holds on future sales.